In [ ]:
# uninstall conflicting hub/transformers/accelerate to avoid mixed versions
!pip uninstall -y huggingface-hub transformers accelerate

# install a compatible set (this provides split_torch_state_dict_into_shards)
!pip install "huggingface-hub==1.4.1" "transformers==4.35.0" "accelerate"

# optional runtime dependencies used by rembg (only needed if you want background removal)
!pip install -U onnxruntime

# (do NOT run requirements.txt yet — we will run it after cloning the repo to keep working dir clean)
!echo "Package installs/upgrades finished. NOW RESTART THE KERNEL (Kernel -> Restart) and then run Cell 2."

In [1]:
import os

# Clone repo only if not exists
if not os.path.exists("TripoSR"):
    !git clone https://github.com/VAST-AI-Research/TripoSR.git

%cd TripoSR

# Install requirements
!pip install -r requirements.txt
!pip install -U onnxruntime

# Version check
import importlib
for p in ("huggingface_hub","transformers","accelerate","onnxruntime","torch"):
    try:
        m = importlib.import_module(p)
        print(f"{p}: {getattr(m,'__version__','unknown')}")
    except Exception as e:
        print(f"{p}: FAILED -> {e}")

Cloning into 'TripoSR'...
remote: Enumerating objects: 161, done.
remote: Counting objects: 100% (95/95), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 161 (delta 63), reused 42 (delta 42), pack-reused 66 (from 1)
Receiving objects: 100% (161/161), 36.71 MiB | 42.33 MiB/s, done.
Resolving deltas: 100% (65/65), done.
/kaggle/working/TripoSR
  Cloning https://github.com/tatsy/torchmcubes.git to /tmp/pip-req-build-pjayrdqm
  Running command git clone --filter=blob:none --quiet https://github.com/tatsy/torchmcubes.git /tmp/pip-req-build-pjayrdqm
  Resolved https://github.com/tatsy/torchmcubes.git to commit 3381600ddc3d2e4d74222f8495866be5fafbace4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached transformers-4.35.0-py3-none-any.whl.metadata (123 kB)
  Using cached huggingface_hub-1.4.1-py3-none-any.whl.metadata (13 kB)
INFO: pip is looking at multiple versions of rembg

In [2]:
import torch, subprocess

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

try:
    print(subprocess.check_output(
        ["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
        text=True))
except:
    print("nvidia-smi not available")

Torch: 2.9.0+cu126
CUDA available: True
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB



In [4]:
import os

# ==== SETTINGS ====
INPUT = "/kaggle/working/TripoSR/examples/marble.png"  # change if needed
OUT_DIR = "/kaggle/working/TripoSR/output/0"
MC_RES = 128
FORMAT = "glb"   # or "obj"
DEVICE = "cuda"  # change to "cpu" if needed
# ==================

# Create output folder
os.makedirs(OUT_DIR, exist_ok=True)

# Check input exists
print("Input exists:", os.path.exists(INPUT))

# Run model
!python run.py "{INPUT}" \
    --no-remove-bg \
    --model-save-format {FORMAT} \
    --mc-resolution {MC_RES} \
    --device {DEVICE}

# Copy result to main working directory
mesh_path = f"{OUT_DIR}/mesh.{FORMAT}"
if os.path.exists(mesh_path):
    os.system(f"cp {mesh_path} /kaggle/working/mesh.{FORMAT}")
    print("Saved to /kaggle/working/mesh." + FORMAT)
else:
    print("Output not found.")

Input exists: True
2026-02-21 00:19:14,549 - INFO - Initializing model ...
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
2026-02-21 00:19:20,964 - INFO - Initializing model finished in 6415.13ms.
2026-02-21 00:19:20,964 - INFO - Processing images ...
2026-02-21 00:19:20,996 - INFO - Processing images finished in 31.95ms.
2026-02-21 00:19:20,996 - INFO - Running image 1/1 ...
2026-02-21 00:19:20,996 - INFO - Running model ...
2026-02-21 00:19:23,250 - INFO - Running model finished in 2253.70ms.
2026-02-21 00:19:23,250 - INFO - Extracting m

In [ ]:
# Run model
!python run.py "{INPUT}" \
    --no-remove-bg \
    --model-save-format {FORMAT} \
    --mc-resolution {MC_RES} \
    --device {DEVICE}

# Copy result to main working directory
mesh_path = f"{OUT_DIR}/mesh.{FORMAT}"
if os.path.exists(mesh_path):
    os.system(f"cp {mesh_path} /kaggle/working/mesh.{FORMAT}")
    print("Saved to /kaggle/working/mesh." + FORMAT)
else:
    print("Output not found.")

In [ ]:
!ls -la /kaggle/working

In [ ]:
!mkdir -p /kaggle/working/TripoSR/output/0
!python run.py /kaggle/working/TripoSR/examples/marble.png --device cuda --no-remove-bg --model-save-format glb --mc-resolution 128

In [ ]:
# show files in current working dir and the working directory
!pwd
!ls -la
# show nested folder if any
!ls -la chairImage || true

In [ ]:
# create the folder that the script expects
